# Iris dataset (intro to machine learning)

multiclass classification problem.
use the attributes of the flower to predict the species of the flower.
lots of comments cause y not.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.datasets import load_iris

#%load_ext autoreload
#%autoreload 2
#matplotlib inline

In [ ]:
#setting the theme of the seaborn plot
sns.set_theme(style="whitegrid", palette="pastel")

Loading in the dataset:

In [ ]:
iris= load_iris() #raw db

In [ ]:
iris.keys() #attribute names

In [ ]:
iris.data.shape #150 entries, 4 attributes

In [ ]:
iris.feature_names #4 features, column names

In [ ]:
print(iris["DESCR"]) #sataset description

In [ ]:
iris.target_names #3 classes to sort into 

In [ ]:
iris["target"] #which entry belongs to which species

In [ ]:
df = pd.DataFrame(data=iris.data, columns=iris.feature_names) 
#creating a dataframe
df.head()

Exploratory Data Analysis

In [ ]:
df.describe() #statistical summary of the data

In [ ]:
col = "sepal length (cm)"
df[col].hist()
plt.suptitle("Sepal Length (cm)")
plt.show()

In [ ]:
col = "sepal width (cm)"
df[col].hist()
plt.suptitle("Sepal Width (cm)")
plt.show()

In [ ]:
col = "petal length (cm)"
df[col].hist() 
plt.suptitle("Petal Length (cm)")
plt.show()

In [ ]:
col = "petal width (cm)"
df[col].hist() 
plt.suptitle("Petal Width (cm)")
plt.show()

In [491]:
df["target"] = iris.target #adding target column to the dataframe
#df.head(55) #check

Relationship of the data features with the target

In [ ]:
df["target_name"] = df["target"].map({0: "setosa", 1: "versicolor", 2: "virginica"}) 
#mapping target values to target names in a new column
df.head() #check again

In [ ]:
sns.relplot(data=df, x="sepal length (cm)", y="target", hue="target_name")
plt.suptitle("Sepal Length relationship with target", y=1.05)
plt.show()

In [ ]:
sns.relplot(data=df, x="sepal length (cm)", y="target", hue="target_name")
plt.suptitle("Sepal Length relationship with target", y=1.05)
plt.show()

In [ ]:
sns.relplot(data=df, x="sepal width (cm)", y="target", hue="target_name")
plt.suptitle("Sepal Width relationship with target", y=1.05)
plt.show()

In [ ]:
sns.relplot(data=df, x="petal length (cm)", y="target", hue="target_name")
plt.suptitle("Petal Length relationship with target", y=1.05)
plt.show()

In [ ]:
sns.relplot(data=df, x="petal width (cm)", y="target", hue="target_name")
plt.suptitle("Petal Width relationship with target", y=1.05)
plt.show()

Pairplots

In [ ]:
sns.pairplot(df, hue="target_name")
#cross relation between all the features + targets

Training the model

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
df_train, df_test = train_test_split(df, test_size=0.25) 
#saving 25% of the data for testing, rest for training 
#shakes up the data so that we get a random sample for training and testing

In [ ]:
df_train.shape, df_test.shape #check the split

In [ ]:
df_train.head()

In [ ]:
df_test.head()

In [ ]:
X_train = df_train.drop(columns=["target", "target_name"])
Y_train = df_train["target"]

### Preparing the data for modelling

okay so since this is a supervised dataset + classification problem, we can use:
1) random guessing
2) manual classification (decison tree-like)
3) logistic regression model

##### Random guessing
3 options -> evenly balanced -> 100/3 = 33.3%
so if our models beat this, they're worth using

#### Manual Classification using sepal length and width
since the split for the sentosa species is obvious on the basis of petal width and petal length, we can use it like a simple if else ladder classification

In [ ]:
def manual(petal_length):
    if petal_length < 2.5:
        return 0
    elif petal_length < 4.8:
        return 1
    else:
        return 2

In [ ]:
prediction = [manual(val) for val in X_train["petal length (cm)"].values]

In [ ]:
result_manual = np.array(prediction) == Y_train
result_manual

In [ ]:
print(f"Manual model accuracy = {np.mean(result_manual)*100:.2f}%")

#### Logistic Regression
this is like linear regression, but not over a continuous value range, but rather classification on the basis of probability that an item belongs to the class. The line is fit using least squares in linear regression but we use maximum likelihood here.

In [ ]:
from sklearn.linear_model import LogisticRegression

Using a validation set to evaluate our model

In [ ]:
model = LogisticRegression()

In [ ]:
Xt,Xv,yt,yv = train_test_split(X_train, Y_train, test_size=0.25)

In [ ]:
model.fit(Xt, yt)

In [ ]:
pred = model.predict(Xv)
pred

In [ ]:
yv.values

In [ ]:
print(f"model accuracy = {np.mean(pred == yv)*100:.2f}%") 
#idk why it's so low esp because only two predictions are off but sure

In [ ]:
model.score(Xv,yv)

#### Cross validation
split the data into sections. a section is chosen as test, rest is used for training. all sections are chosen in the next iterations.

In [ ]:
from sklearn.model_selection import cross_val_score, cross_val_predict

In [ ]:
acc= np.mean(cross_val_score(model, X_train, Y_train, cv=5, scoring="accuracy"))

In [ ]:
print(f"model crossvalidation accuracy = {acc*100:.2f}%") 
#better than the single validation score

#### misclassification of points

In [ ]:
pred = cross_val_predict(model, X_train, Y_train, cv=5)

In [ ]:
#copied the data and added a column of predictions to it
df_predictions = df_train.copy()
df_predictions["prediction"] = pred
df_predictions.head()

In [ ]:
df_predictions["correct predictions"]= pred == Y_train
df_predictions["correct predictions"].values

In [ ]:
incorrect_predictions = ~df_predictions["correct predictions"]
incorrect_predictions.values

In [ ]:
df_predictions["prediction label"] = df_predictions["prediction"].map({0: "setosa", 1: "versicolor", 2: "virginica"})
df_predictions.head()

In [ ]:
sns.scatterplot(x="petal length (cm)", y = "petal width (cm)", hue="prediction label", data=df_predictions)

In [ ]:
sns.scatterplot(data=df_predictions, x= "petal length (cm)", y = "petal width (cm)", hue="target_name")

comparing the scatterplots, the model actuall ypredicted well except for 3 data points which overlap in the areas of classification.

In [ ]:
def plot_incorrect_predictions(df_predictions, x_axis, y_axis):
    fig, axs = plt.subplots(2,2, figsize=(10,10))
    axs = axs.flatten()
    sns.scatterplot(x=x_axis, y=y_axis, hue = "prediction label", data=df_predictions, ax=axs[0])
    sns.scatterplot(x=x_axis, y=y_axis, hue = "target_name", data=df_predictions, ax=axs[1])
    sns.scatterplot(x=x_axis, y=y_axis, hue = "correct predictions", data=df_predictions, ax=axs[2])
    axs[3].set_visible(False)    
plt.show()

In [ ]:
plot_incorrect_predictions(df_predictions, "petal length (cm)", "petal width (cm)")

#### Model Tuning
trying to determine and changing the parameters of our model to get the best outcome.

In [ ]:
for reg in (0.25, 0.5, 1, 1.5, 2, 2.5, 5, 10):
    model = LogisticRegression(max_iter=200, C=reg) #tweaking the regularization parameter to see if it improves accuracy
    acc= np.mean(cross_val_score(model, X_train, Y_train, cv=5, scoring="accuracy"))
    print(f"model accuracy = {acc*100:.2f}% with C={reg}")
#it plateaued lolol so best at 0.5, try Bayesian next

##### Final Model!!!

In [ ]:
model = LogisticRegression(max_iter=200, C=1)

In [ ]:
X_test = df_test.drop(columns=["target", "target_name"])
Y_test = df_test["target"]

In [ ]:
X_test.head()

In [ ]:
Y_test.head()

Training the model on our whole data

In [ ]:
model.fit(X_train, Y_train)

In [ ]:
test_pred = model.predict(X_test)

In [ ]:
print(f"model accuracy = {np.mean(test_pred==Y_test)*100:.2f}%")

In [ ]:
correct_classified= test_pred==Y_test
np.sum(~correct_classified)
#it got only 1 wrong

In [ ]:
df_test_predict = df_test.copy()
df_test_predict["prediction"] = test_pred
df_test_predict["correct"] = correct_classified
df_test_predict["prediction label"] = df_test_predict["prediction"].map({0: "setosa", 1: "versicolor", 2: "virginica"})
df_test_predict.head()

In [ ]:
sns.scatterplot(x="petal length (cm)", y = "petal width (cm)", hue="prediction label", data=df_test_predict)

In [ ]:
sns.scatterplot(x="petal length (cm)", y = "petal width (cm)", hue="target_name", data=df_test_predict)

Honestly, it's a reasonable error since the data point is so close to the Versicolor region.  
Achieved a 97.37% accuracy

_Logistic Regression Parameters_  
penalty 'deprecated'  
C 1  
l1_ratio 0.0  
dual False  
tol 0.0001  
fit_intercept True  
intercept_scaling 1  
class_weight None  
random_state None  
solver 'lbfgs'  
max_iter 200  
verbose 0  
warm_start False  
n_jobs None